<a id="encrypted-math-tutorial-fixed-point"></a>
# Encrypted Math Tutorial: Fixed Point

This tutorial covers `src/concrete_fhe_toolkit/math/fixed_point.py`. FHE only natively supports integers. To compute with fractional numbers, we scale them up (e.g., $2.5 \times 10 = 25$) and track the implicit denominator. The toolkit provides helpers to round, rescale, and multiply these simulated floats.

<a id="encoding-decoding"></a>
## Encoding & Decoding
Translating between Python floats and scaled integers.

In [ ]:
from concrete_fhe_toolkit.math.fixed_point import encode_fixed_point, decode_fixed_point

# Scale of 100 means 2 decimal places of precision
SCALE = 100

val1 = 3.14
enc1 = encode_fixed_point(val1, scale=SCALE)
assert enc1 == 314

dec1 = decode_fixed_point(enc1, scale=SCALE)
assert dec1 == 3.14
print("✅ Encoding/Decoding passed!")

<a id="rounding-operations-make_floor-make_ceil-make_round"></a>
## Rounding Operations (`make_floor`, `make_ceil`, `make_round`)

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.math.fixed_point import make_floor, make_ceil, make_round

floor_fn = make_floor(min_input=-500, max_input=500, scale=SCALE)
ceil_fn = make_ceil(min_input=-500, max_input=500, scale=SCALE)
round_fn = make_round(min_input=-500, max_input=500, scale=SCALE)

def test_rounding(val: int):
    return floor_fn(val), ceil_fn(val), round_fn(val)

assert test_rounding(314) == (3, 4, 3) # floor(3.14)=3, ceil(3.14)=4, round(3.14)=3
assert test_rounding(380) == (3, 4, 4) # round(3.80)=4
print("Cleartext rounding passed!")

compiler = fhe.Compiler(test_rounding, {"val": "encrypted"})
inputset = [(-500,), (0,), (314,), (380,)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(380)
assert enc_res == (3, 4, 4)
print("✅ Encrypted rounding passed!")

<a id="fixed-point-multiplication-make_fixed_point_multiply"></a>
## Fixed Point Multiplication (`make_fixed_point_multiply`)
If you multiply two values scaled by $S$, the result is scaled by $S^2$. This function implicitly divides the product by $S$ so the result remains at scale $S$.

In [ ]:
from concrete_fhe_toolkit.math.fixed_point import make_fixed_point_multiply

mul_fn = make_fixed_point_multiply(scale=SCALE, rounding="nearest")

def test_mul(a: int, b: int):
    return mul_fn(a, b)

# 2.50 * 2.00 = 5.00 -> 250 * 200 / 100 = 500
assert test_mul(250, 200) == 500
print("Cleartext multiplication passed!")

compiler = fhe.Compiler(test_mul, {"a": "encrypted", "b": "encrypted"})
inputset = [(0, 0), (250, 200), (-100, 100)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(250, 200)
assert enc_res == 500
print("✅ Encrypted multiplication passed!")